In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
colleges = [
    'College A', 'College B', 'College C', 'College D', 'College E',
    'College F', 'College G', 'College H', 'College I', 'College J'
]

branches = [
    'CSE', 'ECE', 'ME', 'Civil', 'EEE', 'IT',
    'Biotechnology', 'Chemical', 'Aeronautical', 'Data Science'
]

years = list(range(2018, 2024))  # Academic years

# Define branch-wise realistic cutoff values
branch_cutoffs = {
    'CSE': (85, 95), 'ECE': (80, 90), 'ME': (75, 85), 'Civil': (65, 80),
    'EEE': (75, 85), 'IT': (80, 90), 'Biotechnology': (70, 80),
    'Chemical': (70, 80), 'Aeronautical': (75, 85), 'Data Science': (85, 95)
}

# Define branch-wise placement percentage ranges
branch_placement = {
    'CSE': (70, 100), 'ECE': (60, 95), 'ME': (50, 80), 'Civil': (40, 70),
    'EEE': (60, 90), 'IT': (70, 95), 'Biotechnology': (30, 60),
    'Chemical': (50, 80), 'Aeronautical': (50, 75), 'Data Science': (80, 100)
}

# Generate synthetic data
num_records = len(colleges) * len(branches) * len(years)
data = []

for college in colleges:
    for branch in branches:
        # For each branch and college, start with random values for the first year
        initial_min_placement, initial_max_placement = branch_placement[branch]
        initial_percentile_cutoff = np.round(np.random.uniform(branch_cutoffs[branch][0], branch_cutoffs[branch][1]), 2)

        min_placement_percentage = round(np.random.uniform(initial_min_placement, initial_max_placement),2)
        avg_placement_percentage = round(np.random.uniform(min_placement_percentage, initial_max_placement),2)
        max_placement_percentage = round(np.random.uniform(avg_placement_percentage, initial_max_placement),2)

        # Now generate the data for each year
        for year in years:
            if year != years[0]:
                # Ensure placement increases over the years
                min_placement_percentage = min_placement_percentage + np.random.uniform(1, 5)
                avg_placement_percentage = avg_placement_percentage + np.random.uniform(1, 5)
                max_placement_percentage = max_placement_percentage + np.random.uniform(1, 5)

                # Ensure percentile cutoffs decrease over the years
                initial_percentile_cutoff = initial_percentile_cutoff - np.random.uniform(1, 3)
                initial_percentile_cutoff = max(initial_percentile_cutoff, branch_cutoffs[branch][0])  # Prevent going below minimum

            # Introduce anomalies in placement or percentile (randomly in a few cases)
            if np.random.rand() < 0.05:  # 5% chance of anomaly
                # Introduce a low placement anomaly (unrealistically low for that year)
                min_placement_percentage = np.random.uniform(10, 40)
                avg_placement_percentage = min_placement_percentage + np.random.uniform(1, 10)
                max_placement_percentage = avg_placement_percentage + np.random.uniform(1, 10)

            if np.random.rand() < 0.05:  # 5% chance of anomaly
                # Introduce an unusually high percentile cutoff (unrealistically high for that year)
                initial_percentile_cutoff = np.random.uniform(90, 100)

            # Store the data for the year
            data.append({
                'College': college,
                'Branch': branch,
                'Year': year,
                'Percentile_Cutoff': round(initial_percentile_cutoff, 2),  # Use round() function
                'Min_Placement_Percentage': round(min_placement_percentage, 2),  # Use round() function
                'Avg_Placement_Percentage': round(avg_placement_percentage, 2),  # Use round() function
                'Max_Placement_Percentage': round(max_placement_percentage, 2)  # Use round() function
            })

# Convert to DataFrame
df = pd.DataFrame(data)

# Save to Excel file
df.to_excel('realistic_placement_data_with_anomalies.xlsx', index=False)


In [2]:
df.shape

(600, 7)

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

# Load the dataset (assuming you have saved it as 'realistic_placement_data_with_anomalies.xlsx')
df = pd.read_excel('realistic_placement_data_with_anomalies.xlsx')

# Label encoding for categorical features (College, Branch)
label_encoder_college = LabelEncoder()
df['College_Encoded'] = label_encoder_college.fit_transform(df['College'])

label_encoder_branch = LabelEncoder()
df['Branch_Encoded'] = label_encoder_branch.fit_transform(df['Branch'])

# Features for prediction models
features = ['College_Encoded', 'Branch_Encoded', 'Year']

# Step 2: Train/Test Split (for both placement and cutoff)
X = df[features]
y_placement = df['Avg_Placement_Percentage']
y_cutoff = df['Percentile_Cutoff']

# Split data for both placement and cutoff prediction
X_train, X_test, y_train_placement, y_test_placement = train_test_split(X, y_placement, test_size=0.2, random_state=42)
X_train, X_test, y_train_cutoff, y_test_cutoff = train_test_split(X, y_cutoff, test_size=0.2, random_state=42)

In [5]:
# Train a RandomForestRegressor model for predicting placement percentage
placement_model = RandomForestRegressor(n_estimators=100, random_state=42)
placement_model.fit(X_train, y_train_placement)

# Make predictions on the test set
y_pred_placement = placement_model.predict(X_test)

# Evaluate the model using Mean Absolute Error
mae_placement = mean_absolute_error(y_test_placement, y_pred_placement)
print(f'Mean Absolute Error for Placement Prediction: {mae_placement:.2f}')

Mean Absolute Error for Placement Prediction: 9.24


In [6]:
# Train a RandomForestRegressor model for predicting percentile cutoff
cutoff_model = RandomForestRegressor(n_estimators=100, random_state=42)
cutoff_model.fit(X_train, y_train_cutoff)

# Make predictions on the test set
y_pred_cutoff = cutoff_model.predict(X_test)

# Evaluate the model using Mean Absolute Error
mae_cutoff = mean_absolute_error(y_test_cutoff, y_pred_cutoff)
print(f'Mean Absolute Error for Percentile Cutoff Prediction: {mae_cutoff:.2f}')

Mean Absolute Error for Percentile Cutoff Prediction: 2.41


In [7]:
# Use IsolationForest to detect anomalies
anomaly_detector = IsolationForest(contamination=0.05, random_state=42)  # Set contamination to 5%
df['Anomaly'] = anomaly_detector.fit_predict(df[['Min_Placement_Percentage', 'Avg_Placement_Percentage', 'Max_Placement_Percentage', 'Percentile_Cutoff']])

# Mark anomalies: -1 means anomaly, 1 means normal
df['Anomaly'] = df['Anomaly'].map({1: 'Normal', -1: 'Anomaly'})

# Print rows with detected anomalies
anomalies = df[df['Anomaly'] == 'Anomaly']
print(f'Anomalies detected:\n{anomalies.head()}')

Anomalies detected:
      College        Branch  Year  Percentile_Cutoff  \
53  College A  Aeronautical  2023              75.00   
55  College A  Data Science  2019              89.77   
58  College A  Data Science  2022              85.00   
59  College A  Data Science  2023              85.00   
65  College B           CSE  2023              85.00   

    Min_Placement_Percentage  Avg_Placement_Percentage  \
53                     13.48                     14.90   
55                     21.24                     27.87   
58                     12.83                     19.98   
59                     17.21                     21.07   
65                    111.46                    116.93   

    Max_Placement_Percentage  College_Encoded  Branch_Encoded  Anomaly  
53                     16.26                0               0  Anomaly  
55                     33.40                0               5  Anomaly  
58                     21.62                0               5  Anomaly  
59

In [10]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
import markdown
import datetime

class AdvancedPlacementAnomalyReporter:
    def __init__(self):
        self.data = None
        self.anomalies = None
        self.preprocessor = None
        self.anomaly_detector = None

    def load_data(self, file_path):
        """
        Load placement data from Excel file
        """
        self.data = pd.read_excel(file_path)
        return self

    def preprocess_data(self):
        """
        Prepare data for anomaly detection
        """
        # Create features for anomaly detection
        self.data['Year_Numeric'] = pd.to_numeric(self.data['Year'])

        # Select features for anomaly detection
        features = [
            'Year_Numeric',
            'Percentile_Cutoff',
            'Min_Placement_Percentage',
            'Avg_Placement_Percentage',
            'Max_Placement_Percentage'
        ]

        categorical_features = ['College', 'Branch']

        # Column Transformer for preprocessing
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(),
                 [f for f in features if f != 'Year_Numeric']),
                ('cat', OneHotEncoder(handle_unknown='ignore'),
                 categorical_features)
            ])

        return self

    def detect_anomalies(self, contamination=0.1):
        """
        Detect anomalies using Isolation Forest
        """
        # Prepare full feature set
        X = self.preprocessor.fit_transform(self.data)

        # Anomaly detection using Isolation Forest
        self.anomaly_detector = IsolationForest(
            contamination=contamination,
            random_state=42
        )

        # Predict anomalies
        self.data['Anomaly_Score'] = self.anomaly_detector.fit_predict(X)
        self.data['Is_Anomaly'] = self.data['Anomaly_Score'].map({1: 0, -1: 1})

        # Separate anomalies
        self.anomalies = self.data[self.data['Is_Anomaly'] == 1]

        return self

    def generate_interactive_visualizations(self):
        """
        Create interactive Plotly visualizations
        """
        # Anomalies by Branch Heatmap
        branch_anomalies = self.anomalies.groupby('Branch').size().reset_index()
        branch_anomalies.columns = ['Branch', 'Anomaly_Count']

        fig_branch = px.treemap(
            branch_anomalies,
            path=['Branch'],
            values='Anomaly_Count',
            title='Anomalies Distribution by Branch',
            color='Anomaly_Count',
            color_continuous_scale='Viridis'
        )
        fig_branch.write_html("branch_anomalies_treemap.html")

        # Placement Percentage Boxplot
        fig_placement = go.Figure()
        fig_placement.add_trace(
            go.Box(
                y=self.data[self.data['Is_Anomaly'] == 0]['Avg_Placement_Percentage'],
                name='Normal',
                boxpoints='outliers'
            )
        )
        fig_placement.add_trace(
            go.Box(
                y=self.data[self.data['Is_Anomaly'] == 1]['Avg_Placement_Percentage'],
                name='Anomalies',
                boxpoints='outliers'
            )
        )
        fig_placement.update_layout(
            title='Placement Percentage Distribution: Normal vs Anomalies',
            yaxis_title='Average Placement Percentage'
        )
        fig_placement.write_html("placement_distribution.html")

        return self

    def generate_markdown_report(self):
        """
        Generate a comprehensive Markdown report
        """
        # Calculate summary statistics
        total_records = len(self.data)
        anomaly_count = len(self.anomalies)
        anomaly_percentage = (anomaly_count / total_records) * 100

        # Most anomalous branches
        branch_anomaly_summary = self.anomalies.groupby('Branch').size().nlargest(5)

        # Top anomalous colleges
        college_anomaly_summary = self.anomalies.groupby('College').size().nlargest(5)

        # Prepare Markdown content
        report_content = f"""
# Placement Anomaly Detection Report

## Executive Summary

**Date of Analysis:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

### Key Findings
- **Total Records Analyzed:** {total_records}
- **Anomalies Detected:** {anomaly_count}
- **Anomaly Percentage:** {anomaly_percentage:.2f}%

## Detailed Insights

### Most Anomalous Branches
{branch_anomaly_summary.to_markdown()}

### Top Anomalous Colleges
{college_anomaly_summary.to_markdown()}

### Placement Anomaly Characteristics
- **Minimum Placement Percentage in Anomalies:** {self.anomalies['Avg_Placement_Percentage'].min():.2f}%
- **Maximum Placement Percentage in Anomalies:** {self.anomalies['Avg_Placement_Percentage'].max():.2f}%
- **Average Percentile Cutoff in Anomalies:** {self.anomalies['Percentile_Cutoff'].mean():.2f}

## Recommendations
1. Conduct in-depth investigation of identified anomalous institutions
2. Review placement strategies for most affected branches
3. Develop targeted intervention programs

## Methodology
- **Anomaly Detection Algorithm:** Isolation Forest
- **Contamination Rate:** {self.anomaly_detector.contamination}
- **Random Seed:** 42

*This report is generated using an automated anomaly detection pipeline.*
"""

        # Write Markdown report
        with open('anomaly_report.md', 'w') as f:
            f.write(report_content)

        # Optional: Convert Markdown to HTML
        html_report = markdown.markdown(report_content)
        with open('anomaly_report.html', 'w') as f:
            f.write(html_report)

        return self

    def run_full_analysis(self, file_path, contamination=0.1):
        """
        Execute complete anomaly detection and reporting pipeline
        """
        (self.load_data(file_path)
             .preprocess_data()
             .detect_anomalies(contamination)
             .generate_interactive_visualizations()
             .generate_markdown_report())

        return self.anomalies

# Example usage
if __name__ == "__main__":
    reporter = AdvancedPlacementAnomalyReporter()
    anomalies = reporter.run_full_analysis(
        'realistic_placement_data_with_anomalies.xlsx',
        contamination=0.1
    )
    print("Anomaly Analysis Complete. Check generated reports.")

Anomaly Analysis Complete. Check generated reports.
